## Preprocessing a relational dataset for ClavaDDPM

Berka is eight linked tables, not one flat file. ClavaDDPM still trains a TabDDPM-style model on each table, but it also needs **primary and foreign keys** so parent and child rows can be joined, and a **table graph** so generation can walk from roots to leaves.

This notebook downloads the raw CSVs, then calls `preprocess_berka_all_tables` to write ClavaDDPM inputs under `multi_table/data/berka`. You can preprocess tables independently, but **do not drop ID columns**. Downstream `load_tables` ignores any CSV column whose name contains `_id` when building the model features; those IDs still have to stay in the CSV so tables can be linked.

### Domain files (`{table}_domain.json`)

Each table gets a domain file that lists **modeled features only** (no IDs) and their type. Discrete columns use multinomial diffusion; continuous columns use Gaussian diffusion.

`account.csv` keeps `account_id` and `district_id`. Only the two features appear in the domain:

```json
{
    "frequency": {
        "type": "discrete"
    },
    "account_date": {
        "type": "continuous"
    }
}
```

### Table graph (`dataset_meta.json`)

This file lists every table’s parents and children, plus `relation_order` (the edges ClavaDDPM trains on). After you run the notebook, open `multi_table/data/berka/dataset_meta.json`. For Berka the root is `district`; `disp` has two parents (`client` and `account`); `trans`, `loan`, `order`, and `card` are leaves.

### Outputs

Written under `multi_table/data/berka` (not a train/holdout split — one full CSV per table):

| File | Role |
|------|------|
| `{table}.csv` | Encoded table **including** primary and foreign keys |
| `{table}_domain.json` | Feature types (`discrete` / `continuous`); IDs omitted |
| `dataset_meta.json` | Parent/child graph and `relation_order` |
| `preprocess_meta.json`, `label_encoders.pkl` | Encoders and date epochs (for inverse-transform later) |

### Per-table transforms

Same feature recipe as the single-table pipeline, with IDs kept:

1. Dates (YYMMDD) → integer days since the earliest date in that table (or a shared `date_epoch` if you set one).
2. Client `birth_number` → `year`, `month`, `client_date`, `gender` (month ≥ 50 means female).
3. Missing categoricals → `""` (a lone space is kept as its own level).
4. Discrete columns → integers via `sklearn.preprocessing.LabelEncoder`.

`trans` is large (~1M rows). The call below subsamples **only** `trans` (`trans_sample_size=20000`). Other tables stay in full, so `account_id` foreign keys remain valid. Set that argument to `None` to keep every transaction.

The single-table notebook drops `trans_id` / `account_id` and writes train/holdout splits. This pipeline keeps all relational IDs and writes one CSV per table.

In [1]:
import os
from pathlib import Path


# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/home/coder/synthetic-data-bootcamp')

In [2]:
import pandas as pd

from implementations.tabular_data.multi_table.data_preprocessing.pre_process_berka_all_tabels import (
    preprocess_berka_all_tables,
)
from implementations.tabular_data.utils import download_and_save_multi_table_data

In [3]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
print("IMPLEMENTATION ROOT: ", IMPLEMENTATION_ROOT)
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"
RAW_DATA_PATH = Path(base_data_dir, "raw_data")
# Default dataset
DATASET_NAME = (
    "Berka"  # More details about the Berka data: https://webpages.charlotte.edu/mirsad/itcs6265/group1/domain.html
)

IMPLEMENTATION ROOT:  /home/coder/synthetic-data-bootcamp/implementations/tabular_data


In [4]:
# Download the raw dataset.
download_and_save_multi_table_data(DATASET_NAME, RAW_DATA_PATH)
# Load and inspect one sample table
accounts = pd.read_csv(Path(base_data_dir, "raw_data", "account.csv"))
print(accounts.head())

INFO:implementations.tabular_data.utils:Downloading all the tables of the Berka dataset from https://drive.google.com/drive/folders/14kodt5XsnHYgVlvaJ4R6Qm5WSgvRbVmp?usp=sharing -> /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data
INFO:implementations.tabular_data.utils:Downloading URL: https://drive.google.com/drive/folders/14kodt5XsnHYgVlvaJ4R6Qm5WSgvRbVmp?usp=sharing -> /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data
Retrieving folder contents


Processing file 1AwOngMtpm3EMzZ65JxZhGRbh3Pk9ohPf account.csv
Processing file 10kesnMVxILLefFcET_qvAbVqGbV6LIAL card.csv
Processing file 15DEm6Q7A4DHgMGRUtcv3TWiHf8Y27nUV client.csv
Processing file 1NQIKfggt3gDzHu7nEyomUcShGXj5TBCP disp.csv
Processing file 1X_j2OpImsB_zNhzJel-XZs0YzbthE0V4 district.csv
Processing file 1e5PnVdWrB3_NQ493atd_PZVmHAfRZJoK loan.csv
Processing file 1pLNjgmEkwgdera8tKnvsqbx_aG7hi5vv order.csv
Processing file 1HV5CwSBvtw5UmWRVD-KEVhQPkLGad7CG trans.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1AwOngMtpm3EMzZ65JxZhGRbh3Pk9ohPf
To: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data/account.csv
100%|██████████| 155k/155k [00:00<00:00, 3.99MB/s]
Downloading...
From: https://drive.google.com/uc?id=10kesnMVxILLefFcET_qvAbVqGbV6LIAL
To: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data/card.csv
100%|██████████| 31.6k/31.6k [00:00<00:00, 21.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=15DEm6Q7A4DHgMGRUtcv3TWiHf8Y27nUV
To: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data/client.csv
100%|██████████| 94.8k/94.8k [00:00<00:00, 2.48MB/s]
Downloading...
From: https://drive.google.com/uc?id=1NQIKfggt3gDzHu7nEyomUcShGXj5TBCP
To: /home/coder/synthetic-data-bootcamp/implementations/tab

  account_id;"district_id";"frequency";"date"
0            576;55;"POPLATEK MESICNE";930101
1           3818;74;"POPLATEK MESICNE";930101
2            704;55;"POPLATEK MESICNE";930101
3           2378;16;"POPLATEK MESICNE";930101
4           2632;24;"POPLATEK MESICNE";930102



Download completed


In [6]:
# preprocess all the tables
# It is okay to subsample a table that has no children tables.
preprocess_berka_all_tables(
    input_dir=RAW_DATA_PATH,
    output_dir=base_data_dir,
    trans_sample_size=20000,  # subsample trans only; set None for the full ~1M rows
    seed=42,
)

Loading account from /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data/account.csv ...
  4,500 rows, columns=['account_id', 'district_id', 'frequency', 'date']
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/account.csv
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/account_domain.json
  LabelEncoder classes:
    frequency: ['POPLATEK MESICNE', 'POPLATEK PO OBRATU', 'POPLATEK TYDNE']
  Date epoch (YYMMDD): 930101
Loading card from /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data/card.csv ...
  892 rows, columns=['card_id', 'disp_id', 'type', 'issued']
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/card.csv
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/card_domain.json
  LabelEncoder classes:
    card

{'output_dir': PosixPath('/home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka'),
 'n_rows': {'account': 4500,
  'card': 892,
  'client': 5369,
  'disp': 5369,
  'district': 77,
  'loan': 682,
  'order': 6471,
  'trans': 20000},
 'date_epochs': {'account': '930101',
  'card': '931107',
  'client': None,
  'disp': None,
  'district': None,
  'loan': '930705',
  'order': None,
  'trans': '930116'}}

In [7]:
# Inspect the processed version of the table
accounts = pd.read_csv(Path(base_data_dir, "account.csv"))
accounts.head()

,account_id,district_id,frequency,account_date
0,576,55,0,0
1,3818,74,0,0
2,704,55,0,0
3,2378,16,0,0
4,2632,24,0,1
